# Theme 2: Bug-Fix Quality Over Time

**RQ2:** How does bug-fix *acceptance rate* change over time?  
**RQ3:** How does *time to merge* change over time?  
**RQ4:** How does *patch size* change over time?  
**RQ5:** How does *revision burden* change over time?

Dataset: `mabujadallah/GitHub-Agentic-PR-Dataset`  
Coverage: Dec 2024 – Feb 2026 (15 months)

In [ ]:
%pip install matplotlib seaborn scipy pyarrow fsspec requests

In [ ]:
import sys
sys.path.insert(0, '.')
from analysis_utils import (
    load_fix_prs, load_commits, load_commit_details, build_revision_stats,
    merge_rate, chi_square, mann_whitney, sig_label,
    set_plot_style, save_fig,
    AGENTS, AGENT_COLORS, THEME1_DIR, THEME2_DIR, THEME3_DIR,
)
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')
%matplotlib inline
set_plot_style()

In [ ]:
# ── Load data ────────────────────────────────────────────────────────
df         = load_fix_prs()
agents_df  = df[df['is_agent'] & df['agent'].isin(AGENTS)].copy()
human_df   = df[~df['is_agent']].copy()

# Commit data needed for RQ5
commits   = load_commits()
details   = load_commit_details()
rev_stats = build_revision_stats(df, commits, details)
print('All data loaded.')

## RQ2: How does bug-fix acceptance rate change over time?

In [ ]:
# Monthly merge rate per agent + human baseline
months = sorted(df['month'].unique())
rows   = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = agents_df[(agents_df['month'] == m) & (agents_df['agent'] == agent)]
        row[agent] = round(sub['is_merged'].mean() * 100, 1) if len(sub) >= 5 else None
    h = human_df[human_df['month'] == m]
    row['Human'] = round(h['is_merged'].mean() * 100, 1) if len(h) >= 5 else None
    rows.append(row)
monthly_rate = pd.DataFrame(rows).set_index('month')
print('Monthly merge rate (%):')
print(monthly_rate.to_string())

In [ ]:
# Figure: monthly acceptance rate per agent vs human
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_rate.index, monthly_rate[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.plot(monthly_rate.index, monthly_rate['Human'], 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Merge Rate (%)')
ax.set_ylim(0, 105)
ax.set_title('RQ2: Monthly Bug-Fix Acceptance Rate per Agent vs Human')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq2_monthly_merge_rate', THEME2_DIR)

## RQ3: How does time to merge change over time?

In [ ]:
# Monthly median time-to-merge per agent + human
merged_agents = agents_df[agents_df['is_merged']]
merged_human  = human_df[human_df['is_merged']]
rows_ttm = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = merged_agents[(merged_agents['month'] == m) & (merged_agents['agent'] == agent)]
        row[agent] = round(sub['hours_to_merge'].median(), 2) if len(sub) >= 5 else None
    h = merged_human[merged_human['month'] == m]
    row['Human'] = round(h['hours_to_merge'].median(), 2) if len(h) >= 5 else None
    rows_ttm.append(row)
monthly_ttm = pd.DataFrame(rows_ttm).set_index('month')
print('Monthly median time to merge (hours):')
print(monthly_ttm.to_string())

In [ ]:
# Figure: monthly time to merge
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_ttm.index, monthly_ttm[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.plot(monthly_ttm.index, monthly_ttm['Human'], 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Time to Merge (hours)')
ax.set_title('RQ3: Monthly Median Time to Merge per Agent vs Human')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq3_monthly_time_to_merge', THEME2_DIR)

## RQ4: How does bug-fix patch size change over time?

In [ ]:
# Aggregate total lines added/deleted per PR
pr_size = (
    details.groupby('pr_id')
    .agg(lines_added=('additions', 'sum'), lines_deleted=('deletions', 'sum'))
    .reset_index()
    .rename(columns={'pr_id': 'id'})
)
df_size       = df.merge(pr_size, on='id', how='left')
agents_size   = df_size[df_size['is_agent'] & df_size['agent'].isin(AGENTS)]
human_size    = df_size[~df_size['is_agent']]

rows_size = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = agents_size[(agents_size['month'] == m) & (agents_size['agent'] == agent)]
        row[agent] = round(sub['lines_added'].median(), 1) if len(sub) >= 5 else None
    h = human_size[human_size['month'] == m]
    row['Human'] = round(h['lines_added'].median(), 1) if len(h) >= 5 else None
    rows_size.append(row)
monthly_size = pd.DataFrame(rows_size).set_index('month')
print('Monthly median lines added:')
print(monthly_size.to_string())

In [ ]:
# Figure: monthly patch size (lines added)
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_size.index, monthly_size[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.plot(monthly_size.index, monthly_size['Human'], 's--',
        color=AGENT_COLORS['Human'], linewidth=2.5, label='Human', zorder=5)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Lines Added')
ax.set_title('RQ4: Monthly Median Patch Size (Lines Added) per Agent vs Human')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq4_monthly_patch_size', THEME2_DIR)

## RQ5: How does revision burden change over time?

In [ ]:
# Monthly revision rate: % of merged PRs that had >1 commit
agent_rev = rev_stats[rev_stats['agent'].isin(AGENTS)].copy()
agent_rev['is_revised'] = agent_rev['num_commits'] > 1

rows_rev = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = agent_rev[(agent_rev['month'] == m) & (agent_rev['agent'] == agent)]
        row[agent] = round(sub['is_revised'].mean() * 100, 1) if len(sub) >= 5 else None
    rows_rev.append(row)
monthly_rev = pd.DataFrame(rows_rev).set_index('month')
print('Monthly revision rate (%):')
print(monthly_rev.to_string())

In [ ]:
# Figure: monthly revision rate
fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_rev.index, monthly_rev[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Revision Rate (%)')
ax.set_title('RQ5: Monthly Revision Rate per Agent')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq5_monthly_revision_rate', THEME2_DIR)

In [ ]:
# Monthly median revision lines added (for revised PRs only)
rows_revsize = []
for m in months:
    row = {'month': str(m)}
    for agent in AGENTS:
        sub = agent_rev[(agent_rev['month'] == m) & (agent_rev['agent'] == agent)
                        & (agent_rev['num_commits'] > 1)]
        row[agent] = round(sub['rev_lines_added'].median(), 1) if len(sub) >= 5 else None
    rows_revsize.append(row)
monthly_revsize = pd.DataFrame(rows_revsize).set_index('month')

fig, ax = plt.subplots(figsize=(13, 5))
for agent in AGENTS:
    ax.plot(monthly_revsize.index, monthly_revsize[agent], 'o-',
            color=AGENT_COLORS[agent], label=agent, linewidth=1.8)
ax.axvline('2025-07', color='red', linestyle=':', linewidth=1.5, label='AIDev cutoff')
ax.set_xlabel('Month')
ax.set_ylabel('Median Revision Lines Added')
ax.set_title('RQ5: Monthly Median Revision Effort (Lines Added in Revisions)')
ax.legend()
plt.xticks(rotation=45, ha='right')
fig.tight_layout()
save_fig(fig, 'rq5_monthly_revision_effort', THEME2_DIR)